
# ✅ Notebook 03 — Enrollment & Verification (Multi-Familiar System)
This notebook creates a speaker database using embeddings generated in Notebook 02 and allows verification against enrolled familiar speakers.


In [10]:
import os
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
import joblib
import torch

from speechbrain.pretrained import EncoderClassifier
import webrtcvad

from sklearn.metrics.pairwise import cosine_similarity
import json

ROOT = Path('/content/drive/MyDrive/Speaker_Recognition_System')
RAW = ROOT / 'data/raw'
PROCESSED = ROOT / 'data/processed'
MODEL_DIR = ROOT / 'models'
META = ROOT / 'metadata'

FEATURES = MODEL_DIR / "embeddings.npy"
LABELS = MODEL_DIR / "labels.npy"
CLF_PATH = MODEL_DIR / "ecapa_classifier.pkl"
DB_PATH = META / "familiar_db.json"

META.mkdir(exist_ok=True)

print("✅ Paths Ready")

✅ Paths Ready


In [11]:
vad = webrtcvad.Vad(2)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
speech_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="ecapa_model",
    run_opts={"device": DEVICE},
)
print("✅ ECAPA Voice Model Loaded")


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/ecapa_model/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in ecapa_model.
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/ecapa_model/embedding_model.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["embedding_model"] = /content/ecapa_model/embedding_model.ckpt
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/ecapa_model/mean_var_norm_emb.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["mean_var_norm_emb"] = /content/ecapa_model/mean_var_norm_emb.ckpt
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/ecapa_model/classifier.ckp

✅ ECAPA Voice Model Loaded


In [12]:
def remove_silence(y, sr, frame_ms=30):
    frame_len = int(sr * frame_ms / 1000)
    voiced = []
    for i in range(0, len(y) - frame_len, frame_len):
        frame = y[i:i+frame_len]
        if vad.is_speech((frame * 32768).astype(np.int16).tobytes(), sr):
            voiced.extend(frame)
    return np.array(voiced, dtype=np.float32)


def extract_embedding(file_path):
    y, sr = sf.read(file_path, dtype='float32')
    if y.ndim > 1: y = y.mean(axis=1)
    y = remove_silence(y, sr)

    wav = torch.tensor(y).unsqueeze(0)
    with torch.no_grad():
        emb = speech_model.encode_batch(wav).squeeze().cpu().numpy()
    return emb / (np.linalg.norm(emb) + 1e-9)


In [13]:
if DB_PATH.exists():
    with open(DB_PATH, "r") as f:
        familiar_db = json.load(f)
else:
    familiar_db = {}  # { "name" : [embedding_list] }

print("📌 Familiar Users:", list(familiar_db.keys()))


📌 Familiar Users: []


In [14]:
def enroll_user(name, wav_file):
    emb = extract_embedding(wav_file)

    if name not in familiar_db:
        familiar_db[name] = []

    familiar_db[name].append(emb.tolist())

    with open(DB_PATH, "w") as f:
        json.dump(familiar_db, f, indent=4)

    print(f"✅ {name} enrolled successfully!")


In [15]:
THRESHOLD = 0.65  # tune using Notebook-04

def verify_user(wav_file):
    if len(familiar_db) == 0:
        return "NO_FAMILIARS", 0.0

    emb = extract_embedding(wav_file)
    max_score = 0
    best_user = None

    for name, vectors in familiar_db.items():
        vectors = np.array(vectors)
        scores = cosine_similarity([emb], vectors)[0]
        score = np.max(scores)

        if score > max_score:
            max_score = score
            best_user = name

    if max_score >= THRESHOLD:
        return best_user, float(max_score)
    else:
        return "STRANGER", float(max_score)


In [16]:
enroll_user("Ayush", "/content/drive/MyDrive/Speaker_Recognition_System/data/raw/familiar_Ayush1.wav")

✅ Ayush enrolled successfully!


In [19]:
person, score = verify_user("/content/drive/MyDrive/Speaker_Recognition_System/data/raw/stranger_09.wav")
print("Result:", person,"\n", "Score:", score)


Result: STRANGER 
 Score: 0.12729732289263881
